# Challenge 4: Web Scraping y pipeline de datos.

- **Mini pipeline:** scraping de Books to Scrape → enriquecimiento de autores con una API externa → persistencia en PostgreSQL → consultas → indexación y medición de performance.

## 1. Configuración y conexión a PostgreSQL.

### Conexión a PostgreSQL y verificación de respuesta.
- **Entrada:** credenciales del archivo .env (no se escriben en el código).
- **Salida:** imprime (1,) si la conexión funciona.
- **Entregable que cubre:** prerrequisito de la base de datos (conexión).

In [1]:
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv(override=True)

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

cur = conn.cursor()
cur.execute("SELECT 1;")
print(cur.fetchone())
cur.close()
conn.close()

(1,)


## 2. Scraping de categorías y libros.

### Importación de librerías y definición de funciones auxiliares del scraping.
- **Entrada:** ninguna. 
- **Salida:** constantes y funciones listas para usar.
- **Entregable que cubre:** scraping completo (requisito 3).

In [2]:
import re                         # expresiones regulares (limpiar el precio)
import time                       # pausas entre peticiones
import requests                   # peticiones HTTP
from bs4 import BeautifulSoup     # parseo del HTML
from urllib.parse import urljoin  # URLs absolutas a partir de relativas

BASE = "https://books.toscrape.com/"
HEADERS = {"User-Agent": "challenge4-student-project"}
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def get_soup(url):
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    resp.encoding = "utf-8"
    return BeautifulSoup(resp.text, "html.parser")

def parse_price(text):
    return float(re.sub(r"[^\d.]", "", text))

### Obtención de las categorías y recorrida de todos los libros con su paginación.
- **Entrada:** URL base del sitio.
- **Salida:** lista de diccionarios, uno por libro.
- **Entregable que cubre:** scraping completo (requisito 3).

In [3]:
def get_categories():
    soup = get_soup(BASE + "index.html")
    links = soup.select("div.side_categories ul li ul li a")
    return [(a.get_text(strip=True), urljoin(BASE, a["href"])) for a in links]

def scrape_category(name, url):
    books = []
    while url:
        soup = get_soup(url)
        for pod in soup.select("article.product_pod"):
            a = pod.select_one("h3 a")
            rating_word = pod.select_one("p.star-rating")["class"][1]
            books.append({
                "title": a["title"],
                "price": parse_price(pod.select_one("p.price_color").get_text()),
                "rating": RATING_MAP[rating_word],
                "category": name,
                "url": urljoin(url, a["href"]),
            })
        nxt = soup.select_one("li.next a")
        url = urljoin(url, nxt["href"]) if nxt else None
        time.sleep(0.2)
    return books

### Ejecución del scraping completo y verificación de que no falta nada.
- **Salida esperada:** 50 categorías, 1000 libros y 1000 URLs únicas.

In [4]:
all_books = []
categories = get_categories()
print("categorías:", len(categories))
for name, url in categories:
    cat_books = scrape_category(name, url)
    print(f"{name}: {len(cat_books)}")
    all_books.extend(cat_books)

print("TOTAL:", len(all_books))
print("URLs únicas:", len({b["url"] for b in all_books}))

categorías: 50
Travel: 11
Mystery: 32
Historical Fiction: 26
Sequential Art: 75
Classics: 19
Philosophy: 11
Romance: 35
Womens Fiction: 17
Fiction: 65
Childrens: 29
Religion: 7
Nonfiction: 110
Music: 13
Default: 152
Science Fiction: 16
Sports and Games: 5
Add a comment: 67
Fantasy: 48
New Adult: 6
Young Adult: 54
Science: 14
Poetry: 19
Paranormal: 1
Art: 8
Psychology: 7
Autobiography: 9
Parenting: 1
Adult Fiction: 1
Humor: 10
Horror: 17
History: 18
Food and Drink: 30
Christian Fiction: 6
Business: 12
Biography: 5
Thriller: 11
Contemporary: 3
Spirituality: 6
Academic: 1
Self Help: 5
Historical: 2
Christian: 3
Suspense: 1
Short Stories: 1
Novels: 1
Health: 4
Politics: 3
Cultural: 1
Erotica: 1
Crime: 1
TOTAL: 1000
URLs únicas: 1000
